# Parte Específica: Visualização exploratória 2

**Variável:** `Total_Trans_Ct` (quantidade de transações em 12 meses), comparando clientes que
ficaram com clientes que cancelaram o cartão (`Attrition_Flag`).

**Tipo de gráfico:** violino, escolhido pelo grupo (Luis).

In [1]:
# [IA] Claude Code – importacao das bibliotecas, leitura do dataset tratado e pasta de saida
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

CAMINHO_DADOS = Path("../../out/parte_especifica/bankchurners_tratado.csv")
PASTA_SAIDA = Path("../../out/parte_especifica")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CAMINHO_DADOS)

In [2]:
# [IA] Claude Code – rotulos em portugues para a situacao do cliente e cores fixas por grupo
# As mesmas cores (azul = ficou, vermelho = cancelou) devem ser usadas em todos os graficos do grupo.
ROTULOS_SITUACAO = {"Existing Customer": "Ficou", "Attrited Customer": "Cancelou"}
ORDEM_SITUACAO = ["Ficou", "Cancelou"]
CORES_SITUACAO = {"Ficou": "#4C72B0", "Cancelou": "#C44E52"}

df["Situacao"] = df["Attrition_Flag"].map(ROTULOS_SITUACAO)
assert df["Situacao"].notna().all()
df["Situacao"].value_counts().reindex(ORDEM_SITUACAO)

Situacao
Ficou       8500
Cancelou    1627
Name: count, dtype: int64

In [3]:
# [IA] Claude Code – numeros que acompanham o grafico: quartis de Total_Trans_Ct em cada grupo
resumo = (
    df.groupby("Situacao")["Total_Trans_Ct"]
    .describe()[["count", "min", "25%", "50%", "75%", "max", "mean"]]
    .reindex(ORDEM_SITUACAO)
    .round(1)
)
resumo

,count,min,25%,50%,75%,max,mean
Situacao,,,,,,,
Ficou,8500.0,11.0,54.0,71.0,82.0,139.0,68.7
Cancelou,1627.0,10.0,37.0,43.0,51.0,94.0,44.9


In [4]:
# [IA] Claude Code – grafico de violino (densidade + linhas dos quartis), um violino por grupo
fig, ax = plt.subplots(figsize=(9, 6))
sns.violinplot(
    data=df, x="Situacao", y="Total_Trans_Ct",
    hue="Situacao", order=ORDEM_SITUACAO, palette=CORES_SITUACAO, legend=False,
    inner="quartile",   # linhas tracejadas: Q1, mediana (tracejado longo) e Q3
    cut=0,              # nao desenha densidade alem do menor e do maior valor observado
    ax=ax,
)

# Quantidade de clientes de cada grupo no rotulo do eixo x
qtd = df["Situacao"].value_counts()
ax.set_xticks(range(len(ORDEM_SITUACAO)))
ax.set_xticklabels([f"{s}\n(n = {qtd[s]:,})".replace(",", ".") for s in ORDEM_SITUACAO])

# Legenda das linhas internas (o grafico nao tem legenda de cor porque o eixo x ja nomeia os grupos)
from matplotlib.lines import Line2D
ax.legend(
    handles=[
        Line2D([0], [0], color="black", linestyle=(0, (5, 2)), label="Mediana"),
        Line2D([0], [0], color="black", linestyle=":", label="1o e 3o quartis"),
    ],
    loc="upper right", frameon=True,
)

ax.set_title("Quantidade de transacoes em 12 meses: clientes que ficaram x clientes que cancelaram")
ax.set_xlabel("Situacao do cliente")
ax.set_ylabel("Quantidade de transacoes em 12 meses (Total_Trans_Ct)")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "viz_exploratoria_2_violino_trans_ct.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\luisgx\AppData\Local\Temp\ipykernel_32676\3451860525.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Nota analítica

**O que está sendo mostrado:** O gráfico mostra a variável `Total_Trans_Ct`, que representa a quantidade de transações em 12 meses, comparando os grupos `Ficou` e `Cancelou`, derivados de `Attrition_Flag`. A largura do violino representa a densidade dos valores, ou seja, onde há maior concentração de clientes. As linhas tracejadas indicam o primeiro quartil (Q1), a mediana e o terceiro quartil (Q3).

**Qual insight pode ser extraído:** Na base completa, sem filtros, a mediana de quem cancelou é 43 transações, enquanto a de quem ficou é 71. Portanto, a mediana de quem cancelou fica 28 transações abaixo da mediana de quem ficou. Os dois picos aparecem no grupo `Ficou`, sugerindo uma distribuição bimodal e a existência de dois perfis de clientes ativos, com níveis de uso distintos; os clientes que cancelaram estão mais concentrados em uma faixa mais baixa de transações.

**Por que este tipo de visualização foi escolhido:** O violino foi escolhido em vez do boxplot porque, além de mostrar os quartis, a mediana e a comparação entre os grupos, evidencia a forma da distribuição e torna visíveis os dois picos da distribuição bimodal observada em `Total_Trans_Ct`.